In [ ]:
# SolarShield Tutorial: From CVE Disclosure to Community Alert
# ICS 139W Technical Tutorial, University of California Irvine
#
# Copyright (c) 2026 The SolarShield contributors.
# Released for educational use under the terms of the MIT License.

# SolarShield: From CVE Disclosure to Community Alert

* [Your Name], University of California Irvine, ICS 139W

This tutorial walks through how to detect a newly disclosed solar-inverter vulnerability and match it to the U.S. ZIP codes most likely to be affected, using only public data.

**Learning outcomes**

1. By the end, readers will be able to query the NIST National Vulnerability Database for solar-inverter CVEs and score their severity.
2. Readers will be able to join cyber-severity data with U.S. Census community features to produce a ranked alert list of ZIP codes.

## Contents

1. [Overview](#overview)
2. [Grid Security Impact](#grid-security-impact)
3. [Target Audience](#target-audience)
4. [Background & Prerequisites](#background)
5. [Software Requirements](#software-requirements)
6. [Data Description](#data-description)
7. [Methodology](#methodology)
    - [Step 1: Fetch Solar-Related CVEs from NIST NVD](#step-1)
    - [Step 2: Flag Actively Exploited Vulnerabilities Using CISA KEV](#step-2)
    - [Step 3: Score Cyber Pressure Per Vendor](#step-3)
    - [Step 4: Fetch Community Features from U.S. Census ACS](#step-4)
    - [Step 5: Compute Community Risk Prior Score](#step-5)
    - [Step 6: Match a New CVE to High-Risk ZIP Codes](#step-6)
8. [Results & Discussion](#results)
9. [Limitations & Next Steps](#limitations)
10. [References](#references)

<a name="overview"></a>
## 1. Overview

![SolarShield](https://raw.githubusercontent.com/atifusmani1/solar-shield/main/data/solar-shield-logo-text.png)

A **solar inverter** is the device that converts the DC power produced by rooftop solar panels into the AC power that homes and the utility grid use. Modern inverters are increasingly internet-connected, and academic research shows that an attacker who controls the output of just **2% of Europe's inverters — about 4.5 GW of capacity — can crash grid frequency below 49 Hz and trigger cascading blackouts**. The default password on devices from one of the world's largest inverter manufacturers is `123456`.

**SolarShield** is a research project that predicts which U.S. communities are most likely to contain vulnerable residential solar inverter infrastructure — *before* a single device is scanned. Instead of asking "which exposed inverters can we find right now?", it asks "where is vulnerable solar infrastructure most concentrated?", combining public CVE intelligence with U.S. Census community features to rank ZIP codes by risk.

**This tutorial teaches one slice of SolarShield: going from a freshly disclosed CVE to a ranked list of affected communities.** By the end you will have a working end-to-end pipeline that, given any new solar-inverter vulnerability, produces a top-N list of ZIP codes a utility should alert first.

<a name="grid-security-impact"></a>
## 2. Grid Security Impact

Grid security failures are not theoretical. In **December 2015**, attackers used spear-phishing and remote-access tooling to disconnect substations across western Ukraine, leaving roughly **230,000 customers without electricity** in the middle of winter — the first publicly confirmed cyberattack to cause a blackout. Distributed energy resources expand that attack surface. In Japan, **800 SolarView Compact** monitoring devices were hijacked and used as a beachhead for bank theft, and exposures of that same device grew **350% in the two years after the incident** rather than shrinking.

**What an automated alert system would do.** Utilities, public-power authorities, and emergency managers would receive a **ranked list of ZIPs the day a new CVE drops**, instead of discovering exposure months later through downstream incident reports. The pipeline you build here is the scoring core of that system — once it works end-to-end, scheduling it to poll NVD daily and emit alerts above a CVSS threshold is a small additional step.

<a name="target-audience"></a>
## 3. Target Audience

- **Technical level.** Intermediate Python users who are comfortable with `pandas`. **No prior cybersecurity background is required** — every term used in this tutorial is defined the first time it appears.
- **Prerequisites.** Basic familiarity with HTTP / REST APIs is helpful but not required; each API call is shown explicitly.
- **Use case.** Grid security researchers, utility planners and analysts, emergency managers, and students of critical-infrastructure security who want a reproducible workflow that connects CVE intelligence to community-level exposure.

<a name="background"></a>
## 4. Background & Prerequisites

- **CVE (Common Vulnerabilities and Exposures).** A unique identifier (e.g., `CVE-2024-12345`) for a publicly disclosed software or firmware vulnerability. Maintained by MITRE and indexed by the NIST National Vulnerability Database; one CVE corresponds to one vulnerability.
- **CVSS score (Common Vulnerability Scoring System).** A 0.0–10.0 severity score attached to a CVE. Roughly: 0.0–3.9 *Low*, 4.0–6.9 *Medium*, 7.0–8.9 *High*, 9.0–10.0 *Critical*. This tutorial treats CVSS ≥ 7.0 as the alerting threshold.
- **CISA KEV (Known Exploited Vulnerabilities catalog).** A list, maintained by the U.S. Cybersecurity and Infrastructure Security Agency, of CVEs confirmed to be **actively exploited in the wild**. A CVE in KEV is materially more urgent than one that is merely "high-severity on paper."
- **ZCTA (ZIP Code Tabulation Area).** The U.S. Census Bureau's geographic approximation of a USPS ZIP code. Most ZIPs map cleanly to a ZCTA, but the two are not identical — some ZIPs (PO-box-only, very small areas) have no ZCTA at all.
- **Community risk prior.** SolarShield's term for a *pre-scan* estimate of how likely a given ZCTA is to contain vulnerable rooftop solar infrastructure, based on housing and demographic features. It is a **prior**, not a measurement: it tells you *where to look*, not *what is compromised*.

<a name="software-requirements"></a>
## 5. Software Requirements

The notebook depends on four libraries — they are installed in the next cell.

- **`pandas`** — tabular data manipulation; every step's intermediate output is a DataFrame.
- **`requests`** — HTTP client used to call the NVD, CISA KEV, and U.S. Census ACS endpoints.
- **`matplotlib`** — produces the histograms and bar charts in Steps 1, 3, and 5.
- **`numpy`** — numeric operations supporting the readiness and pressure score calculations.

**Expected runtime: under 10 minutes on a Colab free-tier instance.** Each network call is throttled and bounded, so cold-cache and warm-cache runs both complete inside that budget.

In [ ]:
# Install dependencies. The default line works in Colab and pulls the pinned versions
# directly from GitHub so anyone running the published notebook stays in lockstep with
# the version it was tested against.
!pip install -q -r https://raw.githubusercontent.com/atifusmani1/solar-shield/main/tutorials/requirements.txt

# Fallback for local clones (uncomment if the URL above is unavailable):
# !pip install -q pandas requests matplotlib numpy

### Pipeline configuration

These are the knobs for the whole pipeline. Change `TARGET_STATE` to run on another state; change `MIN_CVSS_FOR_ALERT` to tighten the alerting threshold; extend `SOLAR_VENDORS` if you care about a manufacturer that is not listed.

In [ ]:
import time

import pandas as pd
import requests
import matplotlib.pyplot as plt
import numpy as np

CONFIG = {
    "TARGET_STATE": "CA",
    "SOLAR_VENDORS": ["sungrow", "growatt", "sma", "solaredge", "enphase", "solarview"],
    "NVD_SEARCH_KEYWORDS": ["solar inverter", "photovoltaic"],
    "MIN_CVSS_FOR_ALERT": 7.0,
    "TOP_N_ZIPS": 10,
}

print(f"Configured for state {CONFIG['TARGET_STATE']} with {len(CONFIG['SOLAR_VENDORS'])} vendors tracked.")

<a name="data-description"></a>
## 6. Data Description

| Source | What It Provides | Access | URL |
|--------|------------------|--------|-----|
| NIST NVD | CVE entries with CVSS scores, descriptions, and metadata | Public REST API | [services.nvd.nist.gov](https://services.nvd.nist.gov/rest/json/cves/2.0) |
| CISA KEV | Catalog of CVEs confirmed to be actively exploited in the wild | Public JSON feed | [cisa.gov KEV feed](https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json) |
| U.S. Census ACS 5-year | ZCTA-level housing, income, and demographic estimates | Public REST API | [api.census.gov](https://api.census.gov/data/2022/acs/acs5) |

**NIST NVD.** The National Vulnerability Database is the U.S. government's canonical CVE registry, maintained by NIST. It mirrors MITRE's CVE list and adds CVSS scores, configurations, and references. We use it as the source of truth for *which* solar-inverter vulnerabilities exist and *how severe* they are.

**CISA KEV.** The Known Exploited Vulnerabilities catalog is a curated list of CVEs that the U.S. Cybersecurity and Infrastructure Security Agency has confirmed to be actively exploited. Inclusion in KEV is a much sharper signal than CVSS alone — it means an attack is happening, not just possible.

**U.S. Census ACS.** The American Community Survey 5-year estimates aggregate demographic and housing data over a five-year window for fine-grained geographic units, including ZCTAs. We use it to estimate community-level housing stock that correlates with rooftop solar adoption — single-family share, owner-occupancy, and home value.

<a name="methodology"></a>
## 7. Methodology

The pipeline runs in six steps. Each step has a short *what & why* explanation, a function that does the work, and a small output (a table or a plot) so you can see the result before moving on.

<a name="step-1"></a>
### Step 1: Fetch Solar-Related CVEs from NIST NVD

The NIST National Vulnerability Database (NVD) is the canonical public registry of CVEs. We query its REST API once per keyword in `NVD_SEARCH_KEYWORDS`, dedupe by CVE ID, and pull out four fields that matter downstream: the CVSS v3 base score, the severity label, the published date, and the description (which we mine for vendor mentions in Step 3).

**Why two keywords?** "Solar inverter" catches devices labeled as inverters; "photovoltaic" catches research-paper-style language. NVD's search is a literal substring match, so synonyms do not collapse automatically.

**Why no API key?** NVD allows roughly 5 requests per 30-second window without a key — plenty for this tutorial. We sleep 6 seconds between keyword calls to stay under the limit.

In [ ]:
NVD_API = "https://services.nvd.nist.gov/rest/json/cves/2.0"
NVD_THROTTLE_S = 6  # NVD allows 5 requests / 30s without an API key

def _extract_cvss_v3(metrics):
    """Pull CVSS v3 base score and severity, falling back v3.1 -> v3.0."""
    for key in ("cvssMetricV31", "cvssMetricV30"):
        entries = metrics.get(key) or []
        if entries:
            d = entries[0].get("cvssData", {})
            return d.get("baseScore"), d.get("baseSeverity")
    return None, None

def _match_vendors(description, vendors):
    desc_lower = (description or "").lower()
    return [v for v in vendors if v.lower() in desc_lower]

def fetch_cves(keywords, results_per_page=50, vendors=None):
    """Query NVD once per keyword, dedupe by CVE ID, return a tidy DataFrame."""
    vendors = vendors or CONFIG["SOLAR_VENDORS"]
    rows = {}
    for i, kw in enumerate(keywords):
        if i > 0:
            time.sleep(NVD_THROTTLE_S)
        params = {"keywordSearch": kw, "resultsPerPage": results_per_page}
        resp = requests.get(NVD_API, params=params, timeout=30)
        resp.raise_for_status()
        for item in resp.json().get("vulnerabilities", []):
            cve = item.get("cve", {})
            cve_id = cve.get("id")
            if not cve_id or cve_id in rows:
                continue
            descs = cve.get("descriptions", [])
            description = next((d["value"] for d in descs if d.get("lang") == "en"), "")
            score, severity = _extract_cvss_v3(cve.get("metrics", {}))
            rows[cve_id] = {
                "cve_id": cve_id,
                "published": cve.get("published"),
                "cvss_v3_score": score,
                "cvss_v3_severity": severity,
                "description": description,
                "vendor_keywords_matched": _match_vendors(description, vendors),
            }
    df = pd.DataFrame(rows.values())
    if not df.empty:
        df["published"] = pd.to_datetime(df["published"], errors="coerce")
    return df

In [ ]:
cve_df = fetch_cves(CONFIG["NVD_SEARCH_KEYWORDS"])
print(f"Fetched {len(cve_df)} unique CVEs across keywords {CONFIG['NVD_SEARCH_KEYWORDS']}.")

fig, ax = plt.subplots(figsize=(7, 3.5))
cve_df["cvss_v3_score"].dropna().plot(kind="hist", bins=10, ax=ax, color="#2c7fb8", edgecolor="white")
ax.set_xlabel("CVSS v3 base score")
ax.set_ylabel("CVE count")
ax.set_title("Severity distribution of solar-related CVEs")
plt.tight_layout()
plt.show()

cve_df.head()

<a name="step-2"></a>
### Step 2: Flag Actively Exploited Vulnerabilities Using CISA KEV

A high CVSS score means a vulnerability is dangerous *if* exploited. The CISA KEV catalog tells us which CVEs are *actually* being exploited in the wild — a much sharper signal for prioritization. We pull the full KEV feed (a single JSON file, no key required), build a set of CVE IDs, and tag every solar-related CVE in our frame with a boolean `in_kev`.

Solar CVEs rarely appear in KEV — attackers go after broader-impact targets first — but when one does, it should be promoted to the top of any alert queue.

In [ ]:
KEV_FEED = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

def load_kev():
    """Fetch CISA's KEV catalog and return the set of CVE IDs known to be actively exploited."""
    resp = requests.get(KEV_FEED, timeout=30)
    resp.raise_for_status()
    return {v["cveID"] for v in resp.json().get("vulnerabilities", [])}

In [ ]:
kev_ids = load_kev()
cve_df["in_kev"] = cve_df["cve_id"].isin(kev_ids)

n_in_kev = int(cve_df["in_kev"].sum())
print(f"KEV catalog size: {len(kev_ids):,} CVEs.")
print(f"Solar-related CVEs in KEV: {n_in_kev} of {len(cve_df)}.")

cve_df.loc[cve_df["in_kev"], ["cve_id", "cvss_v3_score", "description"]].head()

<a name="step-3"></a>
### Step 3: Score Cyber Pressure Per Vendor

The CVE list tells us which vendors carry the most risk *as a whole*. For each vendor in `CONFIG["SOLAR_VENDORS"]`, we count the CVEs whose descriptions mention that vendor and combine four signals into a single **cyber pressure score** in [0, 1]:

- **40%** — max CVSS observed (worst single vulnerability)
- **30%** — mean CVSS across the vendor's CVEs (overall severity tendency)
- **20%** — share of the vendor's CVEs that are critical (CVSS ≥ 9.0)
- **10%** — share of the vendor's CVEs that appear in CISA KEV (active exploitation)

Vendor matching here is purely keyword-based and undercounts CVEs that name a product but not the vendor. The full SolarShield pipeline replaces this with NVD's **CPE** (Common Platform Enumeration) strings; for a tutorial, keyword matching is more legible. Expect the chart to be lopsided — real CVE corpora are sparse and concentrate on a few vendors at a time.

In [ ]:
def score_vendor_pressure(cve_df, vendors):
    """Aggregate CVEs by vendor mention and return a 0-1 cyber pressure score per vendor."""
    rows = []
    for vendor in vendors:
        mask = cve_df["vendor_keywords_matched"].apply(lambda lst: vendor in lst)
        sub = cve_df[mask]
        n = len(sub)
        scores = sub["cvss_v3_score"].dropna()
        max_cvss = float(scores.max()) if not scores.empty else 0.0
        mean_cvss = float(scores.mean()) if not scores.empty else 0.0
        critical_count = int((scores >= 9.0).sum())
        kev_count = int(sub["in_kev"].sum()) if "in_kev" in sub else 0
        crit_rate = critical_count / n if n else 0.0
        kev_rate = kev_count / n if n else 0.0
        # Weights sum to 1 and every component is in [0, 1], so the result is too:
        pressure = 0.4 * (max_cvss / 10) + 0.3 * (mean_cvss / 10) + 0.2 * crit_rate + 0.1 * kev_rate
        rows.append({
            "vendor": vendor,
            "cve_count": n,
            "max_cvss": round(max_cvss, 2),
            "mean_cvss": round(mean_cvss, 2),
            "critical_cve_count": critical_count,
            "kev_count": kev_count,
            "cyber_pressure_score": round(pressure, 3),
        })
    return (pd.DataFrame(rows)
            .sort_values("cyber_pressure_score", ascending=False)
            .reset_index(drop=True))

In [ ]:
pressure_df = score_vendor_pressure(cve_df, CONFIG["SOLAR_VENDORS"])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(pressure_df["vendor"], pressure_df["cyber_pressure_score"], color="#d95f0e")
ax.set_ylabel("Cyber pressure score (0–1)")
ax.set_title("Cyber pressure by solar inverter vendor")
ax.set_ylim(0, 1)
for tick in ax.get_xticklabels():
    tick.set_rotation(20)
plt.tight_layout()
plt.show()

pressure_df

<a name="step-4"></a>
### Step 4: Fetch Community Features from U.S. Census ACS

The U.S. Census **American Community Survey (ACS) 5-year estimates** provide reliable demographic and housing data at the ZCTA level. We pull five variables that drive solar adoption:

- **`B25001_001E`** — total housing units
- **`B25003_002E`** — owner-occupied housing units
- **`B25024_002E`** — single-family detached housing units
- **`B25077_001E`** — median home value
- **`B19013_001E`** — median household income

**Why no API key?** Census allows ~500 unauthenticated requests per day. We make exactly one.

**Why fetch nationally and filter?** The Census API does not expose a state hierarchy under ZCTAs (ZIP areas can cross state lines), so we fetch all ~33,000 U.S. ZCTAs in one request — about 2 MB — and filter client-side using the USPS-defined California ZIP boundary (90001–96162). The full SolarShield pipeline runs nationwide; we restrict to one state here to stay inside the tutorial's runtime budget.

In [ ]:
ACS_BASE = "https://api.census.gov/data/2022/acs/acs5"

ACS_VARIABLES = {
    "B25001_001E": "total_housing_units",
    "B25003_002E": "owner_occupied_units",
    "B25024_002E": "single_family_detached_units",
    "B25077_001E": "median_home_value",
    "B19013_001E": "median_household_income",
}

# USPS-defined ZIP boundaries for client-side state filtering. Extend as needed.
STATE_ZCTA_RANGES = {
    "CA": ("90001", "96162"),
}

def fetch_acs_zcta_features(state_abbr):
    """Fetch ACS 5-year features for every ZCTA in the target state. One row per ZCTA."""
    if state_abbr not in STATE_ZCTA_RANGES:
        raise ValueError(f"Add '{state_abbr}' to STATE_ZCTA_RANGES with its ZIP boundary.")
    lo, hi = STATE_ZCTA_RANGES[state_abbr]
    params = {
        "get": ",".join(["NAME"] + list(ACS_VARIABLES.keys())),
        "for": "zip code tabulation area:*",
    }
    resp = requests.get(ACS_BASE, params=params, timeout=120)
    resp.raise_for_status()
    rows = resp.json()
    df = pd.DataFrame(rows[1:], columns=rows[0])
    df = df.rename(columns={"zip code tabulation area": "zcta", **ACS_VARIABLES})
    # Census uses -666666666 as the missing-value sentinel; coerce to NaN.
    for col in ACS_VARIABLES.values():
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[df[col] < -600000, col] = pd.NA
    df = df[(df["zcta"] >= lo) & (df["zcta"] <= hi)].copy()
    df["state"] = state_abbr
    return df.reset_index(drop=True)

In [ ]:
features_df = fetch_acs_zcta_features(CONFIG["TARGET_STATE"])
print(f"Fetched {len(features_df):,} {CONFIG['TARGET_STATE']} ZCTAs from Census ACS.")

features_df[list(ACS_VARIABLES.values())].describe().round(0)

<a name="step-5"></a>
### Step 5: Compute Community Risk Prior Score

Now we turn raw ACS counts into a **solar readiness** score in [0, 1] per ZCTA — the *prior* on whether rooftop solar is likely concentrated there. Three signals drive it:

- **`single_family_share`** — single-family detached / total housing units. Apartments rarely host rooftop arrays.
- **`owner_occupied_share`** — owner-occupied / total housing units. Renters rarely install panels.
- **`high_value_share`** — 1 if the ZCTA's median home value ≥ \$750,000, else 0. Higher-value neighborhoods adopt solar earlier.

Weighted score (matches the spirit of the production pipeline, trimmed to the three components most legible at the tutorial level):

$$\text{solar\_readiness} = 0.45 \cdot \text{sf\_share} + 0.35 \cdot \text{owner\_share} + 0.20 \cdot \text{high\_value\_share}$$

Weights sum to 1 and every component is in [0, 1], so the result is too. Remember: this is a **prior**, not a measurement — high-scoring ZCTAs are *likely* to host vulnerable inverters; the tutorial's payoff step ranks, it does not confirm.

In [ ]:
def compute_solar_readiness(features_df, high_value_threshold=750_000):
    """Compute a 0-1 solar readiness score per ZCTA from ACS housing features."""
    df = features_df.copy()
    total = df["total_housing_units"].fillna(0).clip(lower=1)
    df["single_family_share"] = (df["single_family_detached_units"].fillna(0) / total).clip(0, 1)
    df["owner_occupied_share"] = (df["owner_occupied_units"].fillna(0) / total).clip(0, 1)
    df["high_value_share"] = (df["median_home_value"].fillna(0) >= high_value_threshold).astype(float)
    df["solar_readiness"] = (
        0.45 * df["single_family_share"]
        + 0.35 * df["owner_occupied_share"]
        + 0.20 * df["high_value_share"]
    )
    return df

In [ ]:
community_df = compute_solar_readiness(features_df)

fig, ax = plt.subplots(figsize=(7, 3.5))
community_df["solar_readiness"].plot(kind="hist", bins=30, ax=ax, color="#41ab5d", edgecolor="white")
ax.set_xlabel("Solar readiness score (0–1)")
ax.set_ylabel("ZCTA count")
ax.set_title(f"Solar readiness across {CONFIG['TARGET_STATE']} ZCTAs")
plt.tight_layout()
plt.show()

(community_df[["zcta", "single_family_share", "owner_occupied_share", "high_value_share", "solar_readiness"]]
    .sort_values("solar_readiness", ascending=False)
    .head(10))

<a name="step-6"></a>
### Step 6: Match a New CVE to High-Risk ZIP Codes

This is the payoff step. The function below takes a single CVE — the kind that "just dropped" on NVD this morning — together with the community readiness and vendor pressure tables built in Steps 3 and 5, and returns the top-N ZCTAs ranked by:

$$\text{alert\_score} = \text{vendor\_cyber\_pressure} \times \text{solar\_readiness}$$

Both factors live in [0, 1], so the product does too. A ZCTA scoring high means the affected vendor has accumulated cyber-pressure history *and* the community has the housing profile that concentrates rooftop solar. We then simulate "a new CVE drops" by picking the highest-CVSS solar CVE pulled in Step 1 and feeding it through.

If the CVE's description names no vendor we recognize, the function falls back to using the CVE's own CVSS as the urgency proxy and labels the vendor as `unknown vendor` — the pipeline still produces an actionable list rather than failing closed.

In [ ]:
def alert_for_new_cve(cve_row, community_df, vendor_pressure_df, top_n=10):
    """Rank ZCTAs by alert_score = vendor_cyber_pressure * solar_readiness for a single CVE."""
    matched = cve_row.get("vendor_keywords_matched") or []
    if not matched:
        # Unrecognized vendor — fall back to the CVE's own CVSS as urgency proxy.
        urgency = float(cve_row.get("cvss_v3_score") or 0) / 10
        vendors_label = "unknown vendor"
    else:
        sub = vendor_pressure_df[vendor_pressure_df["vendor"].isin(matched)]
        urgency = float(sub["cyber_pressure_score"].max()) if not sub.empty else 0.0
        vendors_label = ", ".join(matched)

    df = community_df.copy()
    df["cve_id"] = cve_row.get("cve_id")
    df["affected_vendors"] = vendors_label
    df["vendor_cyber_pressure"] = urgency
    df["alert_score"] = df["vendor_cyber_pressure"] * df["solar_readiness"]

    cols = ["cve_id", "zcta", "affected_vendors",
            "vendor_cyber_pressure", "solar_readiness", "alert_score"]
    return (df[cols]
            .sort_values("alert_score", ascending=False)
            .head(top_n)
            .reset_index(drop=True))

In [ ]:
# Simulate a 'just dropped' CVE: the highest-CVSS solar CVE with a recognized vendor.
candidates = (cve_df.dropna(subset=["cvss_v3_score"])
              .sort_values("cvss_v3_score", ascending=False))
candidates = candidates[candidates["vendor_keywords_matched"].apply(bool)]
new_cve = candidates.iloc[0]
print(f"Simulating: {new_cve['cve_id']} (CVSS {new_cve['cvss_v3_score']}, vendors {new_cve['vendor_keywords_matched']})")

alert_df = alert_for_new_cve(new_cve, community_df, pressure_df, top_n=CONFIG["TOP_N_ZIPS"])

(alert_df.style
    .format({"vendor_cyber_pressure": "{:.3f}",
             "solar_readiness": "{:.3f}",
             "alert_score": "{:.3f}"})
    .background_gradient(subset=["alert_score"], cmap="Reds"))

<a name="results"></a>
## 8. Results & Discussion

The pipeline above takes any newly disclosed solar-inverter CVE and returns a ranked list of California ZCTAs that a utility operator should check first. Reading the example output:

- Each row is a ZIP-area-sized community.
- `vendor_cyber_pressure` is the historical severity baseline for the affected vendor (Step 3).
- `solar_readiness` is the housing/income-derived prior on rooftop-solar concentration (Step 5).
- `alert_score = vendor_cyber_pressure × solar_readiness` — bounded in [0, 1] — gives a single number to triage by.

**What this means for an operator.** Today, a utility security analyst learning about a new solar-inverter CVE has to manually reason about which territories might be affected, then go pull customer or installer data. The pipeline above replaces that reasoning with a one-line call, scoped to the operator's service territory by changing `TARGET_STATE`.

**How it relates to the full SolarShield system.** The dashboard below visualizes exactly this same scoring across the entire United States, with the same `community_risk_prior_score` formula running daily on a freshly synced CVE corpus.

![SolarShield dashboard](https://raw.githubusercontent.com/atifusmani1/solar-shield/main/data/solar-shield-interface.png)

*The full SolarShield dashboard visualizes this same scoring nationwide.*

**Key limitation: this is a prior, not a measurement.** The score predicts where vulnerable inverters are *likely* concentrated, not where specific devices are compromised. Confirming a particular device requires either a device-discovery layer (Shodan/Censys, omitted from this tutorial because both require paid keys) or direct utility data. The alert list is a triage tool for *where to look first*, never a finding that "ZIP X is compromised."

In [ ]:
# Re-render the Step 6 alert as a captioned summary table for the report.
(alert_df.style
    .format({"vendor_cyber_pressure": "{:.3f}",
             "solar_readiness": "{:.3f}",
             "alert_score": "{:.3f}"})
    .background_gradient(subset=["alert_score"], cmap="Reds")
    .set_caption(f"Top {CONFIG['TOP_N_ZIPS']} alert ZIPs for {new_cve['cve_id']}"))

<a name="limitations"></a>
## 9. Limitations & Next Steps

This tutorial is intentionally narrow; the full SolarShield system covers more ground. Honest limitations of the version you just ran:

- **ZCTA ≠ ZIP.** ZCTAs approximate USPS ZIP codes, but a small number of ZIPs (PO-box-only, very small, or very rural areas) have no ZCTA at all. Communities in those ZIPs are silently dropped from the alert list.
- **Keyword vendor matching undercounts.** Step 3 matches vendors by literal substring against CVE descriptions. Many CVEs name the *product* (e.g., "iSolarCloud") without the vendor (Sungrow), so they go uncounted. The full SolarShield pipeline replaces keyword matching with NVD's structured **CPE** (Common Platform Enumeration) strings, which carry vendor and product as separate fields.
- **One state only.** The Census pull is restricted to California to stay inside the tutorial's runtime budget. The production pipeline runs nationwide on the full ~33,000 ZCTAs.
- **Equity considerations.** Community features like single-family share, owner-occupancy, and median home value correlate with income and race. An alerting system that uses these as a *prior* must be careful not to systematically deprioritize lower-income or majority-renter neighborhoods, where compromised inverters are equally — possibly more — consequential. The score should drive *which alerts get sent first*, never *which alerts get ignored*.

**Next steps — the system change.** The next natural extension is a small **daemon that polls NVD daily** and automatically emits alerts when a new solar-related CVE crosses the CVSS threshold (`MIN_CVSS_FOR_ALERT`). The pipeline you just ran is the scoring core of that system. Wiring the polling and notification layer is straightforward once this scoring runs reliably; the hard part — connecting CVE intelligence to community-level priors — is what this tutorial teaches.

<a name="references"></a>
## 10. References

- **NIST National Vulnerability Database.** [nvd.nist.gov](https://nvd.nist.gov/) — Official U.S. government CVE registry with CVSS scoring.
- **CISA Known Exploited Vulnerabilities Catalog.** [cisa.gov/known-exploited-vulnerabilities-catalog](https://www.cisa.gov/known-exploited-vulnerabilities-catalog) — Federally maintained list of actively exploited CVEs.
- **U.S. Census American Community Survey (ACS) 5-Year Estimates.** [census.gov/programs-surveys/acs](https://www.census.gov/programs-surveys/acs) — Community-level housing and demographic estimates.
- **Forescout, "SUN:DOWN: New Vulnerabilities in Solar Power Systems Exposed" (2025).** [forescout.com SUN:DOWN report](https://www.forescout.com/blog/grid-security-new-vulnerabilities-in-solar-power-systems-exposed/) — 46 vulnerabilities across the top 3 inverter manufacturers.
- **Rule, A., et al. (2019). "Ten Simple Rules for Reproducible Research in Jupyter Notebooks." *PLOS Computational Biology*, 15(7), e1007007.** [doi.org/10.1371/journal.pcbi.1007007](https://doi.org/10.1371/journal.pcbi.1007007) — Notebook authoring guide this tutorial follows.
- **SolarShield repository.** [github.com/atifusmani1/solar-shield](https://github.com/atifusmani1/solar-shield) — The full nationwide pipeline that this tutorial slices into one teachable workflow.